# UniSat — AX.25 Live Demo

**What this notebook shows.** Build a real AX.25 UI frame, feed it byte-by-byte through the streaming decoder, visualise the wire format, and verify the FCS — all with the same Python code that runs in the ground station.

**Run it.** No special setup; only the stdlib + the `utils.ax25` module that ships with the repo.

---

### Agenda

1. Encode a beacon
2. Dissect the frame byte-by-byte
3. Feed it into the streaming decoder one byte at a time
4. Verify CRC-16/X.25 matches the canonical RFC-style oracle
5. Fuzz the decoder on random garbage and confirm it never crashes

In [ ]:
# Put the ground-station utils on the import path.
import pathlib
import sys

REPO = pathlib.Path.cwd().resolve()
while REPO.name != 'unisat' and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'ground-station'))

from utils.ax25 import (
    Address,
    encode_ui_frame,
    decode_ui_frame,
    bit_unstuff,
    fcs_crc16,
    Ax25Decoder,
    AX25Error,
)
print('utils.ax25 loaded from', REPO / 'ground-station' / 'utils' / 'ax25.py')

## 1. Build a beacon

AX.25 UI frame structure:

```
[0x7E] [Dst 7B] [Src 7B] [0x03] [0xF0] [info ≤256B] [FCS 2B LE] [0x7E]
```

`Dst` here is the wildcard `CQ-0` (AMSAT broadcast convention). `Src` is the satellite's callsign plus SSID.

In [ ]:
dst = Address('CQ', 0)
src = Address('UN8SAT', 1)

# Simulate a 48-byte beacon payload with a recognisable pattern so we can
# spot it in the hex dump below.
info = bytes(range(48))

frame = encode_ui_frame(dst, src, pid=0xF0, info=info)
print(f'frame length: {len(frame)} bytes')
print(frame.hex())

## 2. Dissect the frame

Everything between the two `0x7E` flags is bit-stuffed — any run of five consecutive `1` bits gets a `0` bit inserted. To read the logical fields we first bit-unstuff, then parse.

In [ ]:
body = bit_unstuff(frame[1:-1])

print('field              bytes')
print(f'  start flag       {frame[0]:02X}')
print(f'  dst (shifted)    {body[0:7].hex()}')
print(f'  src (shifted)    {body[7:14].hex()}')
print(f'  control          {body[14]:02X}  (0x03 = UI frame)')
print(f'  pid              {body[15]:02X}  (0xF0 = no L3)')
print(f'  info (48B)       {body[16:-2].hex()}')
print(f'  fcs (LE)         {body[-2:].hex()}')
print(f'  end flag         {frame[-1]:02X}')

## 3. Stream it through the decoder

`Ax25Decoder` is designed to consume one byte at a time, modelling exactly how a real UART ISR feeds the ground-station radio. It returns `None` for intermediate bytes and a fully decoded `UiFrame` the moment the closing flag arrives.

In [ ]:
decoder = Ax25Decoder()
emitted = []

for byte in frame:
    out = decoder.push_byte(byte)
    if out is not None:
        emitted.append(out)

print(f'decoder fired {len(emitted)} time(s)')
pkt = emitted[0]
print(f'  dst        : {pkt.dst}')
print(f'  src        : {pkt.src}')
print(f'  pid        : 0x{pkt.pid:02X}')
print(f'  info[:8]   : {pkt.info[:8].hex()}')
print(f'  fcs_valid  : {pkt.fcs_valid}')

## 4. CRC oracle

The FCS uses CRC-16/X.25 — the canonical test vector is `fcs_crc16(b'123456789') == 0x906E`. Anyone can sanity-check the whole chain with this one-liner.

In [ ]:
assert fcs_crc16(b'123456789') == 0x906E
print(f'fcs_crc16("123456789") = 0x{fcs_crc16(b"123456789"):04X}  ✓')

## 5. Fuzz the decoder

Requirement **REQ-AX25-014** demands that the decoder never crashes on arbitrary byte sequences — only typed errors are allowed. The cell below runs 10 000 random byte streams through the decoder; the success criterion is that the loop completes without raising anything but `AX25Error`.

In [ ]:
import random

rng = random.Random(0xDEADBEEF)
decoder = Ax25Decoder()
crashes = 0

for _ in range(10_000):
    for _ in range(rng.randint(0, 200)):
        try:
            decoder.push_byte(rng.randint(0, 255))
        except AX25Error:
            pass  # typed errors are allowed by the spec
        except Exception:  # pragma: no cover
            crashes += 1

print(f'10 000 random byte streams, total crashes: {crashes}')
assert crashes == 0

## 6. Going further

- Run `./scripts/verify.sh` to execute the full C + Python test battery that backs this notebook.
- Open `docs/TECHNICAL_DOCUMENTATION.md` §18 for the design details.
- See `docs/tutorials/ax25_walkthrough.md` for a hand-annotated beacon.

Everything above runs identically on the ground station, in CI, and (mod TCP substrate) on the satellite itself.